# 04. AI와 대화하다 — **협업형**

## 이 노트북의 위치

```
기반     : NB01 (듣다)
확장형   : NB02 (시각) + NB03 (음악)
협업형 ✦ : NB04 (AI와 대화) ← 여기
통합     : NB05 (무대)
```

**확장**은 연주자가 주도하는 일방향 변환(소리 → 새 차원)입니다.  
**협업**은 다릅니다 — AI가 **응답**하고 연주자가 그 응답을 **수용·거부·비교**합니다.

피아노 협업에서 AI 음악 생성의 품질 한계를 고려해, 여기서는 **AI가 음을 생성하지 않는** 두 가지 협업 형태에 집중합니다:

- **Mode B — 해석의 대화**: AI가 수치를 읽고 연주자의 습관을 짚어줌. 연주자는 "AI가 맞게 읽었나?"를 판단.
- **Mode C — 시각의 대화**: AI가 곡 성격에 맞는 cue 매핑을 제안. 연주자는 NB02의 자기 매핑과 **비교**.

둘 다 **학생의 직관과 AI의 휴리스틱이 만나는 지점**이 핵심 체험입니다.

**입력**: `artifacts/analysis/*.json`, `artifacts/cues/*.json`  
**출력**: `artifacts/responses/*_interpretive.md`, `mapping_suggestions.json`

In [ ]:
!pip install -q numpy

import warnings
warnings.filterwarnings('ignore')
print('✅ 준비 완료')

In [ ]:
from pathlib import Path
import json
import numpy as np

ARTIFACTS = Path('artifacts')
(ARTIFACTS / 'responses').mkdir(parents=True, exist_ok=True)

pieces = {
    'satie': {
        'title': 'Satie — Gymnopédie No.1',
        'analysis': ARTIFACTS / 'analysis' / 'satie_analysis.json',
        'cues': ARTIFACTS / 'cues' / 'satie_visual_cues.json',
    },
    'prokofiev': {
        'title': 'Prokofiev — Toccata Op.11',
        'analysis': ARTIFACTS / 'analysis' / 'prokofiev_analysis.json',
        'cues': ARTIFACTS / 'cues' / 'prokofiev_visual_cues.json',
    },
}

for key, p in pieces.items():
    for req in ['analysis', 'cues']:
        if not p[req].exists():
            raise FileNotFoundError(f"{p[req]} 없음. NB01-02를 먼저 실행하세요.")
    print(f"✅ {p['title']}")

---
## Mode B — 해석의 대화

분석 JSON의 **수치를 구체적으로 인용**하며 연주자의 습관을 짚는 템플릿 응답.

Basic Pitch 채보의 한계(velocity 수렴, IOI 지터)를 반영해 임계값을 완화했습니다. 수치는 **상대 기준**으로 읽어야 합니다.

In [ ]:
def classify(value, thresholds, labels):
    for t, lbl in zip(thresholds, labels[:-1]):
        if value < t:
            return lbl
    return labels[-1]


def interpret_performance(profile: dict, piece_name: str) -> str:
    """Basic Pitch의 한계를 반영한 임계값 사용."""
    lines = [f'# {piece_name} — 해석의 대화', '']
    
    # 1. 텍스처
    density = profile['density_per_sec']
    texture = classify(density, [3, 7, 12], ['매우 얇은 텍스처', '얇은 텍스처', '두꺼운 텍스처', '매우 밀집된 텍스처'])
    lines.append(f'## 전체 성격')
    lines.append(f'- 음표 밀도: **{density} notes/sec** — {texture}')
    
    # 2. 다이내믹 (Basic Pitch 한계 반영)
    v_mean = profile['velocity_mean']
    v_std = profile['velocity_std']
    dyn_level = classify(v_mean, [50, 62, 75], ['여리게 추정 (p 계열)', '중저 다이내믹 (mp)', '중강 다이내믹 (mf)', '강하게 추정 (f 계열)'])
    dyn_range = classify(v_std, [8, 14, 20], ['좁은 편차', '온건한 편차', '넓은 편차', '매우 넓은 편차'])
    lines.append(f'- 추정 velocity 평균 **{v_mean}** (σ={v_std}) — {dyn_level}, {dyn_range}')
    lines.append(f'  *※ Basic Pitch 채보 기준, 실제 다이내믹은 원음 참조*')
    
    # 3. 루바토
    ioi_cv = profile['ioi_cv']
    rubato = classify(ioi_cv, [0.4, 0.8, 1.3], ['매우 안정된 펄스', '온건한 시간 조율', '명확한 루바토', '극단적 시간 변동'])
    lines.append(f'- IOI 변동계수 **{ioi_cv}** — {rubato}')
    
    # 4. 음역
    spread = profile['register_spread_semitones']
    range_desc = classify(spread, [24, 48, 72], ['2옥타브 이내', '2-4옥타브', '4-6옥타브', '6옥타브+'])
    lines.append(f'- 음역 **{profile["pitch_range"]}** ({spread}반음) — {range_desc}')
    lines.append('')
    
    # 5. 구간별 흐름
    segments = profile['segments']
    if len(segments) >= 2:
        lines.append('## 구간별 흐름')
        densities = [s['density_per_sec'] for s in segments]
        vels = [s['avg_velocity'] for s in segments]
        
        d_diffs = [abs(densities[i] - densities[i-1]) for i in range(1, len(densities))]
        if d_diffs:
            peak_i = d_diffs.index(max(d_diffs)) + 1
            s_prev, s_curr = segments[peak_i-1], segments[peak_i]
            direction = '급증' if s_curr['density_per_sec'] > s_prev['density_per_sec'] else '급감'
            lines.append(f'- 밀도 최대 변화: **{s_prev["time_range"]} → {s_curr["time_range"]}**에서 {s_prev["density_per_sec"]}→{s_curr["density_per_sec"]} notes/s로 {direction}')
        
        v_diffs = [abs(vels[i] - vels[i-1]) for i in range(1, len(vels))]
        if v_diffs and max(v_diffs) > 3:
            peak_i = v_diffs.index(max(v_diffs)) + 1
            s_prev, s_curr = segments[peak_i-1], segments[peak_i]
            direction = 'crescendo 경향' if s_curr['avg_velocity'] > s_prev['avg_velocity'] else 'diminuendo 경향'
            lines.append(f'- 다이내믹 최대 변화: **{s_prev["time_range"]} → {s_curr["time_range"]}**에서 velocity {s_prev["avg_velocity"]}→{s_curr["avg_velocity"]} ({direction})')
        
        max_i = densities.index(max(densities))
        min_i = densities.index(min(densities))
        lines.append(f'- 텍스처 절정: **{segments[max_i]["time_range"]}** — {segments[max_i]["density_per_sec"]} notes/s')
        lines.append(f'- 가장 희소한 순간: **{segments[min_i]["time_range"]}** — {segments[min_i]["density_per_sec"]} notes/s')
    
    # 6. 연주자에게
    lines.append('')
    lines.append('## 연주자에게')
    if ioi_cv > 0.8:
        lines.append(f'- 온셋 타이밍 변동이 큽니다 (CV={ioi_cv}). 지터도 포함되지만 실제 루바토 경향이 있을 가능성이 높습니다.')
    elif ioi_cv < 0.3:
        lines.append(f'- 온셋이 매우 규칙적입니다 (CV={ioi_cv}). 정박 곡이거나 기계적 연주입니다.')
    
    if density < 3:
        lines.append(f'- 매우 낮은 텍스처 밀도({density}/s). 각 음표의 **무게**가 중요한 연주입니다.')
    elif density > 8:
        lines.append(f'- 매우 높은 텍스처 밀도({density}/s). 개별 음보다 **전체 흐름과 에너지**가 중요한 연주입니다.')
    
    lines.append('')
    lines.append('---')
    lines.append('**이 해석이 당신의 연주 의도와 맞나요? 어긋난다면 어디가 다른가요?**')
    
    return '\n'.join(lines)


for key in ['satie', 'prokofiev']:
    profile = json.loads(pieces[key]['analysis'].read_text())
    interpretation = interpret_performance(profile, pieces[key]['title'])
    out_md = ARTIFACTS / 'responses' / f'{key}_interpretive.md'
    out_md.write_text(interpretation)
    print(interpretation)
    print('\n' + '='*60 + '\n')

---
## Mode B 확장 — 두 곡 비교 대화

In [ ]:
def compare_performances(prof_a, name_a, prof_b, name_b) -> str:
    lines = [f'# {name_a} vs {name_b} — 비교 해석', '']
    metrics = [
        ('density_per_sec', 'notes/sec', '음표 밀도'),
        ('velocity_mean', '', '평균 velocity'),
        ('velocity_std', '', '다이내믹 편차'),
        ('ioi_cv', '', '루바토 지표'),
        ('register_spread_semitones', '반음', '음역'),
        ('total_notes', '개', '전체 음표 수'),
    ]
    lines.append('## 수치 대비')
    lines.append('')
    lines.append(f'| 지표 | {name_a} | {name_b} | 비율 |')
    lines.append(f'|---|---|---|---|')
    for key, unit, label in metrics:
        a, b = prof_a[key], prof_b[key]
        ratio = f'{b/a:.1f}x' if a > 0 else 'N/A'
        lines.append(f'| {label} | {a}{unit} | {b}{unit} | {ratio} |')
    lines.append('')
    lines.append('## 관찰')
    d_ratio = prof_b['density_per_sec'] / max(prof_a['density_per_sec'], 0.01)
    lines.append(f'- **텍스처 대비**: {name_b}가 {name_a}보다 {d_ratio:.1f}배 밀집된 텍스처.')
    r_diff = prof_b['register_spread_semitones'] - prof_a['register_spread_semitones']
    if abs(r_diff) > 12:
        wider = name_b if r_diff > 0 else name_a
        lines.append(f'- **음역 대비**: {wider}가 {abs(r_diff)}반음 ({abs(r_diff)//12}옥타브) 더 넓은 음역 사용')
    if prof_a['ioi_cv'] > 0.5 and prof_b['ioi_cv'] < 0.5:
        lines.append(f'- **시간 해석**: {name_a}가 더 자유롭게 시간을 씀 (CV {prof_a["ioi_cv"]} vs {prof_b["ioi_cv"]})')
    elif prof_b['ioi_cv'] > 0.5 and prof_a['ioi_cv'] < 0.5:
        lines.append(f'- **시간 해석**: {name_b}가 더 자유롭게 시간을 씀 (CV {prof_b["ioi_cv"]} vs {prof_a["ioi_cv"]})')
    return '\n'.join(lines)


prof_s = json.loads(pieces['satie']['analysis'].read_text())
prof_p = json.loads(pieces['prokofiev']['analysis'].read_text())
comparison = compare_performances(prof_s, pieces['satie']['title'], prof_p, pieces['prokofiev']['title'])
print(comparison)
out_md = ARTIFACTS / 'responses' / 'compare_interpretive.md'
out_md.write_text(comparison)
print(f'\n💾 {out_md}')

---
## Mode C — 시각의 대화

AI가 곡 수준 특성(onset rate, 음역, IOI CV)으로 **결정 트리**를 타고 cue 매핑을 제안합니다.

학생은 **NB02에서 자기가 만든 매핑과 비교**해보세요. 같으면 직관이 휴리스틱과 일치한 것이고, 다르면 어느 쪽이 음악적으로 더 설득력 있는지 판단하세요.

In [ ]:
def summarize_cues(cues_path: Path) -> dict:
    data = json.loads(cues_path.read_text())
    frames = data['frames']
    summary = {}
    for ch in ['energy','density','onset_density','register','brightness','attack','register_spread','harmonic_tension','velocity_variance']:
        if ch in frames[0]:
            vals = np.array([f[ch] for f in frames])
            summary[ch] = {'mean': float(vals.mean()), 'std': float(vals.std()), 'max': float(vals.max())}
    return summary


def suggest_mapping(stats: dict, profile: dict) -> dict:
    onset_rate = profile['density_per_sec']
    register_semis = profile['register_spread_semitones']
    ioi_cv = profile['ioi_cv']
    is_fast = onset_rate > 7.0
    is_wide_range = register_semis > 60
    is_rubato = ioi_cv > 1.0
    
    if is_fast and is_wide_range:
        mapping = {
            'camera_shake': 'energy', 'particle_count': 'onset_density',
            'bar_height': 'register_spread', 'sphere_scale': 'velocity_variance',
            'bloom': 'attack', 'bg_saturation': 'harmonic_tension',
            'sphere_hue': 'register', 'rotation_speed': 'onset_density',
            'torus_pulse': 'pulse',
        }
        mood = ['explosive', 'relentless', 'percussive']
        rationale = f'onset rate {onset_rate}/s (빠름), 음역 {register_semis}반음 (넓음). 타격 중심 매핑.'
    elif not is_fast and is_rubato:
        mapping = {
            'camera_shake': 'velocity_variance', 'particle_count': 'harmonic_tension',
            'bar_height': 'density', 'sphere_scale': 'energy',
            'bloom': 'brightness', 'bg_saturation': 'register',
            'sphere_hue': 'register_spread', 'rotation_speed': 'energy',
            'torus_pulse': 'pulse',
        }
        mood = ['floating', 'contemplative', 'rubato']
        rationale = f'onset rate {onset_rate}/s (느림), IOI CV {ioi_cv} (루바토). 정적·화성 중심 매핑.'
    else:
        mapping = {
            'camera_shake': 'energy', 'particle_count': 'onset_density',
            'bar_height': 'attack', 'sphere_scale': 'brightness',
            'bloom': 'energy', 'bg_saturation': 'harmonic_tension',
            'sphere_hue': 'register', 'rotation_speed': 'register_spread',
            'torus_pulse': 'pulse',
        }
        mood = ['balanced', 'dynamic', 'expressive']
        rationale = f'onset rate {onset_rate}/s, 음역 {register_semis}반음. 균형형 매핑.'
    
    if stats.get('harmonic_tension', {}).get('mean', 0) > 0.2:
        mood.append('harmonically-tense')
    
    return {
        'mapping': mapping, 'rationale_ko': rationale, 'mood_words': mood,
        'decision': {'onset_rate': onset_rate, 'is_fast': is_fast, 'is_wide_range': is_wide_range, 'is_rubato': is_rubato},
    }


suggestions = {}
for key in ['satie', 'prokofiev']:
    stats = summarize_cues(pieces[key]['cues'])
    profile = json.loads(pieces[key]['analysis'].read_text())
    sug = suggest_mapping(stats, profile)
    suggestions[key] = sug
    print(f"🎨 {pieces[key]['title']}")
    print(f"   판단: {sug['decision']}")
    print(f"   mood: {sug['mood_words']}")
    print(f"   근거: {sug['rationale_ko']}")
    print(f"   매핑:")
    for k, v in sug['mapping'].items():
        print(f'      {k:18s} ← {v}')
    print()

out_json = ARTIFACTS / 'responses' / 'mapping_suggestions.json'
out_json.write_text(json.dumps(suggestions, ensure_ascii=False, indent=2))
print(f'💾 {out_json}')

---
## 협업의 핵심 질문

- Mode B: "AI가 내 연주에서 정말 중요한 것을 짚었나?"
- Mode C: "AI 매핑과 내 매핑, 어느 쪽이 이 곡에 더 맞나? 왜?"

AI의 제안을 **검증·거부·수용**하는 경험이 진짜 협업입니다.

---

**→ NB05 (`05_stage.ipynb`)**: 확장과 협업의 모든 산출물을 하나의 무대로 엮습니다.